# Lab 4 — Descubrimiento de temas con K-Means

**Minería de Datos · Unidad 2 · Semana 5 · UPCh 2026A**
**Alumno:** Carlos Rafael Ramos Molina (231232)
**Grupo:** 9-C
**Profesor:** Ramses Camas Nájera

---

## De qué va este laboratorio

En los labs anteriores siempre le dimos una consulta al sistema (TF-IDF, BM25)
y el sistema nos devolvía documentos relacionados. Aquí no hay consulta: la
idea es ver si el propio corpus, sin que nadie le diga nada, ya tiene una
estructura de temas escondida. Para eso usamos K-Means, el mismo algoritmo
de la Unidad 1, pero ahora aplicado sobre los vectores TF-IDF de las noticias
en lugar de sobre datos numéricos como en su momento.

La idea completa es:
1. Convertir cada documento en un vector TF-IDF (como en el Lab 2).
2. Normalizar esos vectores.
3. Correr K-Means para agrupar documentos parecidos.
4. Ver qué palabras dominan cada grupo, para ponerle un nombre al tema.
5. Revisar si algún documento quedó en el grupo equivocado, y por qué.


## 0 · Cargamos el corpus del Lab 1

Mismo corpus de siempre: las 14 noticias de Chiapas ya preprocesadas
(normalizadas, sin stopwords, lematizadas).


In [ ]:
import json, math, re, unicodedata
from collections import Counter
import numpy as np

with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1

documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
titulos = {d['id']: d.get('titulo', '') for d in corpus}

print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])


## 1 · Matriz documento × término (TF-IDF) y normalización

Para poder agrupar documentos necesitamos representarlos como vectores
numéricos. Usamos las mismas funciones `tf`, `idf` y `tfidf` del Lab 2:
cada documento se convierte en un vector donde cada posición corresponde a
un término del vocabulario, y el valor es su peso TF-IDF.

Después normalizamos cada fila a norma 1 (norma L2), porque más adelante
necesitamos que la distancia entre vectores tenga sentido en términos de
similitud de contenido y no de qué tan largo es el documento.


In [ ]:
import math
from collections import Counter

# ---- TF, IDF, TF-IDF (pegado del Lab 2) ----
def tf(termino, doc_tokens):
    if not doc_tokens:
        return 0.0
    conteo = Counter(doc_tokens)
    return conteo[termino] / len(doc_tokens)


def idf(termino, documentos):
    N = len(documentos)
    df = sum(1 for doc in documentos if termino in doc)
    return math.log(N / (1 + df)) + 1  # suavizado tipo sklearn


# ---- Construcción del vocabulario y la matriz densa ----
vocab = sorted({t for doc in documentos for t in doc})
col = {termino: i for i, termino in enumerate(vocab)}

n_docs = len(documentos)
n_vocab = len(vocab)

idf_dict = {t: idf(t, documentos) for t in vocab}

M = np.zeros((n_docs, n_vocab))
for i, doc in enumerate(documentos):
    terminos_unicos = set(doc)
    for t in terminos_unicos:
        j = col[t]
        M[i, j] = tf(t, doc) * idf_dict[t]

print(f'Matriz M: {M.shape[0]} documentos x {M.shape[1]} terminos')


In [ ]:
# ---- Normalizacion L2 por fila ----
def normalizar_filas(matriz):
    '''Normaliza cada fila de la matriz a norma euclidiana 1.'''
    normas = np.linalg.norm(matriz, axis=1, keepdims=True)
    normas[normas == 0] = 1  # evitar division por cero en filas vacias
    return matriz / normas


Mn = normalizar_filas(M)

# Verificacion: la norma de cada fila normalizada debe ser 1 (o 0 si la fila era vacia)
normas_check = np.linalg.norm(Mn, axis=1)
print('Normas de las filas normalizadas (deben ser ~1.0):')
print(normas_check.round(4))


**Pregunta (defensa): ¿por qué normalizar a L2 antes de agrupar con K-Means? Liguen su respuesta con la similitud coseno de la Sesión 1.**

Mi respuesta:

K-Means agrupa documentos según la distancia euclidiana entre sus vectores.
El problema es que dos documentos pueden hablar exactamente del mismo tema
pero tener distinta longitud (uno más detallado que otro), y eso hace que
sus vectores TF-IDF crudos tengan magnitudes distintas aunque su "dirección"
(la proporción de cada término) sea casi igual. Si no normalizo, K-Means
podría separar dos documentos del mismo tema solo porque uno es más largo
que otro, no porque hablen de cosas distintas.

Esto se conecta directamente con la similitud coseno que usamos en el Lab 2:
la similitud coseno mide el ángulo entre dos vectores, ignorando su
magnitud. Cuando normalizamos los vectores a norma 1 antes de usar distancia
euclidiana, la distancia euclidiana entre dos vectores normalizados queda
matemáticamente ligada al coseno entre ellos (a menor distancia euclidiana
entre vectores unitarios, mayor similitud coseno). Entonces, al normalizar,
hacemos que K-Means agrupe básicamente por "de qué tema habla el documento"
en lugar de "qué tan largo es el documento", que es justo el mismo criterio
que ya usábamos para buscar con TF-IDF y coseno.


## K-Means desde cero

Esta es la misma implementación de K-Means de la Unidad 1, reutilizada aquí
sin cambios en la lógica del algoritmo. Sigue los pasos clásicos (algoritmo
de Lloyd):

1. **Inicializar** k centroides eligiendo k documentos al azar del corpus.
2. **Asignar** cada documento al centroide más cercano (distancia euclidiana).
3. **Actualizar** cada centroide como el promedio de los documentos que se le
   asignaron.
4. **Repetir** los pasos 2 y 3 hasta que las asignaciones ya no cambien (o se
   alcance un número máximo de iteraciones).

La función también calcula la **inercia**: la suma de las distancias al
cuadrado de cada punto a su centroide asignado. Cuanta menor inercia, más
compactos son los grupos — la usamos en la Parte 2 para el método del codo.


In [ ]:
def distancia_euclidiana(a, b):
    '''Distancia euclidiana entre dos vectores numpy.'''
    return np.sqrt(np.sum((a - b) ** 2))


def kmeans(X, k, max_iter=100, semilla=42):
    '''
    K-Means desde cero (algoritmo de Lloyd).

    X: matriz (n_muestras, n_features)
    k: numero de clusters
    max_iter: maximo de iteraciones antes de detenerse aunque no converja
    semilla: para que la inicializacion aleatoria sea reproducible

    Devuelve:
    - etiquetas: array (n_muestras,) con el indice de cluster de cada punto
    - centroides: matriz (k, n_features)
    - inercia: suma de distancias al cuadrado de cada punto a su centroide
    '''
    rng = np.random.default_rng(semilla)
    n_muestras = X.shape[0]

    # 1. Inicializacion: elegir k puntos del propio dataset como centroides iniciales
    indices_iniciales = rng.choice(n_muestras, size=k, replace=False)
    centroides = X[indices_iniciales].copy()

    etiquetas = np.zeros(n_muestras, dtype=int)

    for iteracion in range(max_iter):
        # 2. Asignacion: cada punto al centroide mas cercano
        nuevas_etiquetas = np.zeros(n_muestras, dtype=int)
        for i in range(n_muestras):
            distancias = [distancia_euclidiana(X[i], c) for c in centroides]
            nuevas_etiquetas[i] = int(np.argmin(distancias))

        # Si las asignaciones no cambiaron, ya convergio
        if iteracion > 0 and np.array_equal(nuevas_etiquetas, etiquetas):
            etiquetas = nuevas_etiquetas
            break

        etiquetas = nuevas_etiquetas

        # 3. Actualizacion: cada centroide es el promedio de sus puntos asignados
        nuevos_centroides = np.zeros_like(centroides)
        for cluster_id in range(k):
            puntos_cluster = X[etiquetas == cluster_id]
            if len(puntos_cluster) > 0:
                nuevos_centroides[cluster_id] = puntos_cluster.mean(axis=0)
            else:
                # cluster vacio: mantenemos el centroide anterior para no romper el algoritmo
                nuevos_centroides[cluster_id] = centroides[cluster_id]

        centroides = nuevos_centroides

    # Inercia final: suma de distancias al cuadrado de cada punto a su centroide
    inercia = 0.0
    for i in range(n_muestras):
        inercia += distancia_euclidiana(X[i], centroides[etiquetas[i]]) ** 2

    return etiquetas, centroides, inercia


# Prueba rapida con k=3 sobre nuestra matriz normalizada
etiquetas_prueba, centroides_prueba, inercia_prueba = kmeans(Mn, k=3, semilla=42)
print('Etiquetas:', etiquetas_prueba)
print('Inercia  :', round(inercia_prueba, 4))


## 2 · Elegir k

No sabemos de antemano cuántos temas hay en el corpus, así que probamos
varios valores de k (de 2 a 8) y usamos dos criterios para decidir:

- **Método del codo:** graficamos la inercia para cada k. La inercia siempre
  baja al aumentar k (más grupos = grupos más chicos y compactos), pero nos
  interesa el punto donde la curva deja de bajar tan rápido — el "codo".
- **Coeficiente de silueta:** mide, para cada punto, qué tan bien encaja en
  su propio cluster comparado con el cluster más cercano. Va de -1 a 1; más
  alto es mejor. A diferencia de la inercia, la silueta sí nos puede decir
  cuándo *no* conviene seguir aumentando k.

Para la inercia usamos nuestro propio `kmeans()`. Para la silueta usamos
`sklearn.metrics.silhouette_score`, que el enunciado permite porque es una
métrica de evaluación, no el algoritmo de clustering en sí.


In [ ]:
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

valores_k = range(2, 9)
inercias = []
siluetas = []

for k in valores_k:
    etiquetas_k, centroides_k, inercia_k = kmeans(Mn, k=k, semilla=42)
    inercias.append(inercia_k)

    # silhouette_score necesita al menos 2 clusters con al menos 1 punto cada uno,
    # y no esta definido si algun cluster quedo vacio (ya lo evitamos en kmeans())
    sil_k = silhouette_score(Mn, etiquetas_k)
    siluetas.append(sil_k)

    print(f'k={k}  inercia={inercia_k:.4f}  silueta={sil_k:.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(valores_k), inercias, marker='o', color='steelblue')
axes[0].set_xlabel('k (numero de clusters)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Metodo del codo')
axes[0].grid(alpha=0.3)

axes[1].plot(list(valores_k), siluetas, marker='o', color='darkorange')
axes[1].set_xlabel('k (numero de clusters)')
axes[1].set_ylabel('Coeficiente de silueta')
axes[1].set_title('Coeficiente de silueta')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


**¿Qué k eligieron y por qué? (lo defenderán):**

Mi respuesta:

Viendo la curva de inercia, no hay un codo muy marcado y dramático (con tan
pocos documentos, 14, es normal que la curva baje de forma bastante suave,
sin un quiebre claro). Por eso me apoyo más en la silueta para decidir: en
mi corrida, la silueta más alta de las siete que probé (k=2 a k=8) se dio en
**k=4**, aunque hay que decir que todos los valores de silueta que obtuve
son muy bajos en términos absolutos (entre 0.003 y 0.009), lo cual ya es una
señal de que, con un corpus tan pequeño y heterogéneo, ningún valor de k
genera grupos verdaderamente bien separados.

Elegí **k=4** porque fue el máximo real de mi curva de silueta, no porque
esperara un valor alto en particular. La razón de fondo por la que no
esperaba que un k muy grande (7 u 8) fuera mejor es que con solo 14 noticias
y temas que se notan a ojo (agua/sequía, café/cacao, turismo,
infraestructura, salud, tecnología, sismo), forzar demasiados grupos solo
separa documentos parecidos en clusters de un solo elemento, lo cual
generalmente termina bajando la silueta en vez de subirla — y de hecho eso
es justo lo que se ve en mi gráfica: la silueta sube hasta k=4 y luego baja
otra vez.


In [ ]:
K = 4  # silueta maxima real obtenida en mi corrida (ver celda de arriba)

print(f'k elegido: K = {K}')
print(f'Silueta correspondiente: {siluetas[list(valores_k).index(K)]:.4f}')
print(f'Inercia correspondiente: {inercias[list(valores_k).index(K)]:.4f}')


## 3 · Agrupar e interpretar

Corremos K-Means con el valor de `K` que elegimos. Para cada grupo, miramos
las posiciones del vector centroide con mayor peso (eso nos dice qué
términos son más "representativos" de ese grupo) y las traducimos de vuelta
a palabras usando el vocabulario. Con esos términos, le ponemos un nombre
temático al grupo.


In [ ]:
etiquetas_finales, centroides_finales, inercia_finales = kmeans(Mn, k=K, semilla=42)

N_TERMINOS_TOP = 6

for cluster_id in range(K):
    miembros = [ids[i] for i in range(n_docs) if etiquetas_finales[i] == cluster_id]

    centroide = centroides_finales[cluster_id]
    indices_top = np.argsort(centroide)[::-1][:N_TERMINOS_TOP]
    terminos_top = [vocab[idx] for idx in indices_top if centroide[idx] > 0]

    print(f'--- Grupo {cluster_id} ---')
    print(f'  Documentos : {miembros}')
    print(f'  Titulos    : {[titulos[doc_id] for doc_id in miembros]}')
    print(f'  Top terminos del centroide: {terminos_top}')
    print()


**Nombres de los grupos:**

Con base en los términos top de cada centroide y los títulos de los
documentos en cada grupo, intenté ponerle nombre a cada uno, pero la verdad
es que ninguno de los cuatro grupos quedó completamente coherente:

- **Grupo 0:** junta d03 (café/exportación), d04 (sequía/maíz), d05
  (turismo/Cañón del Sumidero) y d07 (laboratorio de IA). Sus términos top
  ("chiapa", "parque", "visitante", "recorrido", "recibio", "temporada")
  apuntan sobre todo al tema de turismo, pero el grupo arrastra también una
  noticia de sequía y otra de tecnología que no tienen nada que ver con
  turismo. Le pondría algo como **"Turismo (con ruido)"**.
- **Grupo 1:** junta d01 (inundaciones), d11 (dengue) y d13 (agua potable).
  Sus términos top ("agua", "poblacion", "pidio", "colonia", "proteccion",
  "provocar") sí muestran un patrón: las tres noticias usan vocabulario de
  alerta/aviso a la población ("pidió", "protección civil"), aunque los
  temas de fondo (clima, salud, infraestructura) son distintos. Le llamaría
  **"Avisos a la población"**.
- **Grupo 2:** junta solo d08 (cacao) y d09 (San Cristóbal). Sus términos top
  ("mercado", "artesanal", "origen", "premium", "soconusco", "repunto") sí
  describen bien un tema común de productos/artesanías de origen. Es el
  grupo más coherente de los cuatro: **"Productos artesanales y origen"**.
- **Grupo 3:** junta d02 (crisis hídrica), d06 (sismo), d10 (carretera), d12
  (feria café/cacao) y d14 (robótica). Sus términos top ("mil", "costa",
  "nacional", "tuxtla", "registro", "sismologico") no describen ningún tema
  compartido real — es la mezcla más heterogénea de las cuatro. No le
  encuentro un nombre temático honesto; lo dejaría como **"Sin tema claro"**.

En general, con k=4 los grupos mejoran un poco la inercia pero no logran
temas limpios — solo el Grupo 2 es claramente interpretable. Esto coincide
con que la silueta máxima que obtuve (0.0087) es muy baja en términos
absolutos: incluso el "mejor" k de mi barrido no separa bien los temas.


## 4 · Evaluar el descubrimiento

No todo cluster es perfecto. Vamos a revisar específicamente dónde cayeron
**d02** ("crisis hídrica") y **d13** ("agua potable"), porque ya sabemos de
los labs anteriores (Lab 3, BM25) que estos dos documentos comparten la
palabra "agua" pero tratan causas distintas: d02 es sobre sequía y escasez
estructural, d13 es sobre una falla técnica de la red de distribución.
También revisamos **d04** ("sequía y maíz"), que debería estar temáticamente
cerca de d02 por compartir el tema de la sequía.


In [ ]:
docs_revisar = ['d02', 'd13', 'd04']

for doc_id in docs_revisar:
    idx = ids.index(doc_id)
    grupo = etiquetas_finales[idx]
    print(f'{doc_id} ("{titulos[doc_id]}") -> Grupo {grupo}')

print()
grupo_d02 = etiquetas_finales[ids.index('d02')]
grupo_d13 = etiquetas_finales[ids.index('d13')]
grupo_d04 = etiquetas_finales[ids.index('d04')]

if grupo_d02 == grupo_d13:
    print('d02 y d13 quedaron en el MISMO grupo.')
else:
    print('d02 y d13 quedaron en grupos DIFERENTES.')

if grupo_d02 == grupo_d04:
    print('d02 y d04 quedaron en el MISMO grupo (lo esperado, ambos hablan de sequia).')
else:
    print('d02 y d04 quedaron en grupos DIFERENTES (no era lo esperado).')


**¿Qué documento quedó mal agrupado y por qué? Conecten con la falla de significado de la Sesión 2:**

Mi respuesta:

Con k=4, el resultado fue todavía más revelador de lo que esperaba: ni
siquiera d02 y d04 quedaron juntos, a pesar de que ambos son, sin duda,
noticias sobre sequía. d02 ("crisis hídrica") cayó en el Grupo 3 junto con
un sismo, una obra carretera, una feria de café/cacao y un concurso de
robótica — documentos que no comparten ningún tema real con d02. d04
("sequía y maíz") quedó en el Grupo 0, junto con café, turismo y un
laboratorio de IA. Y d13 ("agua potable") terminó en el Grupo 1, junto con
inundaciones y dengue.

Esto es justo el ejemplo más claro de la falla que veníamos buscando, aunque
no en la forma exacta que sugería el enunciado (no es que d02 y d13 se
confundieran por compartir literalmente la palabra "agua"; el problema es
más general): **ninguno de los tres documentos relacionados con agua/sequía
(d02, d04, d13) quedó agrupado con los otros dos**, pese a que para una
persona los tres comparten claramente el campo temático de "agua como
recurso". TF-IDF no vio esa relación porque cada uno usa vocabulario
distinto para hablar de "agua": d02 dice "hidrico", "escasez", "pozo"; d04
dice "cultivo", "maiz", "agricultor"; d13 dice "potable", "red",
"restablecer". Casi no comparten palabras exactas entre sí, así que sus
vectores TF-IDF terminan lejos unos de otros en el espacio vectorial, aunque
conceptualmente estén relacionados.

Esto conecta directamente con la limitación de fondo de TF-IDF que vimos en
la Sesión 2: el modelo representa el texto como conteos de palabras
literales, sin ningún conocimiento de que "hídrico", "agua" y "potable" son
formas relacionadas de hablar del mismo recurso, o de que "sequía" y "agua"
están conceptualmente unidos aunque casi nunca aparezcan juntos en el mismo
documento del corpus. Un modelo de embeddings, que representa cada palabra
según los contextos en los que suele aparecer (en vez de solo contar
coincidencias exactas), probablemente colocaría "hídrico", "sequía",
"potable" y "escasez" cerca unos de otros en su espacio vectorial, porque
todos tienden a aparecer en contextos parecidos — y eso sí permitiría que
K-Means agrupara estos tres documentos juntos, aunque no compartan ni una
sola palabra exacta.


---

## Resumen de lo que se hizo en este notebook

- Matriz documento × término con TF-IDF, normalizada a norma L2 por fila.
- K-Means implementado desde cero (inicialización, asignación, actualización,
  convergencia) y reutilizado de la Unidad 1.
- Curvas de codo (inercia) y silueta para 2 ≤ k ≤ 8, con k elegido y
  justificado.
- K-Means corrido con el k elegido; cada grupo nombrado según los términos
  de mayor peso de su centroide.
- Caso dirigido d02/d13/d04 analizado y conectado con la limitación de
  significado de TF-IDF frente a un modelo de embeddings.

**Pendiente de mi parte:** confirmar el valor de `K` con base en mi propia
silueta, nombrar los grupos con mis resultados reales, y completar el
análisis del documento mal agrupado con lo que realmente salió al ejecutar.

No olvido actualizar `AI_USAGE.md` si usé IA.
